### **Step 1: Request files from CartoSpot API and read them into Dataframes**
#### Enter the ID to get the required INTERVAL Dataset after running the cell:

In [2]:
import os
import sys
import zipfile
from io import BytesIO
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import geojson
import geopandas as gpd
from shapely.geometry import Point

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent
OUTPUTS = ROOT / "outputs"

# Get the data Id from user input
print("----")
data_id = input("Enter the data Id for your INTERVAL Dataset (e.g., 306): ")
# data_id = 306

# Validate user input (ensure it's a positive integer)
try:
    data_id = int(data_id)
    if data_id <= 0:
        raise ValueError("Data Id must be a positive integer.")
except ValueError as e:
    print(f"Invalid input: {e}")
    sys.exit(1)  # Exit with error code 1

# Define the API endpoint and file paths
api_endpoint = f"https://api.cartospot.com/api/v2/db/cartospot/generate-csv/interval/{data_id}/"
csv_folder = str(OUTPUTS / "csv_files")
places_file_path = os.path.join(csv_folder, "places.csv")
submissions_file_path = os.path.join(csv_folder, "submissions.csv")

# Ensure the folder for CSV files exists
if not os.path.exists(csv_folder):
    os.makedirs(csv_folder)

# Request the API endpoint and extract files
try:
    response = requests.get(api_endpoint)
    if response.status_code == 200:
        # Assume the response is a ZIP file
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            z.extractall(csv_folder)
        print("----")
        print(f"Extracted all files to '{csv_folder}'.")

        # Read the extracted CSV files into DataFrames
        try:
            places_df = pd.read_csv(places_file_path)
            submissions_df = pd.read_csv(submissions_file_path)
            print("----")
            print("CSV files loaded into DataFrames successfully!")
        except Exception as e:
            print(f"Unexpected error reading CSV files: {e}")
            sys.exit(2)  # Exit with error code 2
    else:
        print(f"Failed to download files. Status code: {response.status_code}")
        sys.exit(3)  # Exit with error code 3
except Exception as e:
    print(f"Error fetching and extracting files: {e}")
    sys.exit(4)  # Exit with error code 4

----


Enter the data Id for your INTERVAL Dataset (e.g., 306):  342


----
Extracted all files to 'outputs/csv_files'.
----
CSV files loaded into DataFrames successfully!


### **Step 2: Preview Places and Submissions raw data**
#### Run cell below:

In [3]:
# View Places dataframe data types
print(places_df.info())
# View first Places 10 rows
print(places_df.head(10))

# View Submissions dataframe data types
print(submissions_df.info())
# View first 10 Submissions rows
print(submissions_df.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                120 non-null    int64  
 1   unique_id         0 non-null      float64
 2   location_type     120 non-null    object 
 3   tree_privacy      120 non-null    object 
 4   NTM_ID            108 non-null    object 
 5   tree_height       108 non-null    float64
 6   avg_tree_height   108 non-null    float64
 7   canopy_perimeter  108 non-null    float64
 8   canopy_area       108 non-null    float64
 9   STD               108 non-null    float64
 10  geometry_x        120 non-null    float64
 11  geometry_y        120 non-null    float64
 12  dataset_id        120 non-null    int64  
 13  dataset_name      120 non-null    object 
 14  url               120 non-null    object 
 15  submitter_name    113 non-null    object 
 16  submitter_id      108 non-null    float64
 1

### **Step 3: Clean and merge data to export final CSVs, GeoJSON and SHP files**
#### Run the cell below:

In [4]:
# Replace 'Other' in species with other_species data if applicable
submissions_df['species'] = np.where(
    (submissions_df['species'] == 'Other') & (submissions_df['other_species'].notna()),
    submissions_df['other_species'],
    submissions_df['species']
)

# Reduce and rename columns for Places DataFrame
places_df_trans = places_df[
    ['Id', 'location_type', 'tree_privacy', 'NTM_ID', 'tree_height', 'avg_tree_height', 'canopy_perimeter', 'canopy_area', 'geometry_x', 'geometry_y', 'dataset_id', 'dataset_name', 'url']
].rename(columns={
    'Id': 'location_id',
    'url': 'location_url'
})

# Reduce and rename columns for Submissions DataFrame
submissions_df_trans = submissions_df[
    ['Id', 'location_id', 'does_a_tree_exist_here', 'species', 'tree_type', 'health', 'circumference', 'dbh', 'notes', 'image_url', 'submitter_name']
].rename(columns={
    'Id': 'submission_id'
})

# Select the latest submission for each location_id using 'Id' as the incremental identifier
latest_submissions_df = submissions_df_trans.sort_values(by=['location_id', 'submission_id']).drop_duplicates(subset='location_id', keep='last')


# Merge Places with the latest Submissions
merged_df = pd.merge(
    places_df_trans,
    latest_submissions_df,
    on='location_id',
    how='left'
)

# Save the DataFrames to CSV files
output_folder_csv = str(OUTPUTS / "csv_files")
os.makedirs(output_folder_csv, exist_ok=True)

data_all_file = os.path.join(output_folder_csv, f"data_all_places_dataset_id_{data_id}.csv")
merged_df.to_csv(data_all_file, index=False)

print(f"Saved all places data to '{data_all_file}'.")

# Create GeoJSON file
output_folder_geofiles = str(OUTPUTS / "geofiles")
os.makedirs(output_folder_geofiles, exist_ok=True)

geojson_file = os.path.join(output_folder_geofiles, f"data_all_places_dataset_id_{data_id}.geojson")
features = [
    geojson.Feature(
        id=row['location_id'],
        geometry=geojson.Point((row['geometry_x'], row['geometry_y'])),
        properties=row.drop(['geometry_x', 'geometry_y', 'location_id']).to_dict()
    )
    for _, row in merged_df.fillna('').iterrows()
]

geojson_data = geojson.FeatureCollection(features)
with open(geojson_file, 'w') as f:
    geojson.dump(geojson_data, f)
print("----")
print(f"GeoJSON file created: {geojson_file}")

# Create Shapefile
shapefile_path = os.path.join(output_folder_geofiles, f"data_all_places_dataset_id_{data_id}.shp")
gdf = gpd.GeoDataFrame(
    merged_df.fillna(''),
    geometry=[Point(xy) for xy in zip(merged_df['geometry_x'], merged_df['geometry_y'])],
    crs="EPSG:4326"
).drop(columns=['geometry_x', 'geometry_y']).rename(columns={
    'location_id': 'loc_id',
    'location_type': 'loc_type',
    'tree_privacy': 'privacy',
    'tree_height': 'height',
    'avg_tree_height': 'avg_height',
    'canopy_perimeter': 'canopy_per',
    'canopy_area': 'canopy_area',
    'dataset_name': 'dataset',
    'location_url': 'loc_url',
    'submission_id': 'sub_id',
    'does_a_tree_exist_here': 'tree_exist',
    'circumference': 'circumf',
    'submitter_name': 'submitter'
})
gdf.to_file(shapefile_path, driver="ESRI Shapefile")
print("----")
print(f"Shapefile created: {shapefile_path}")


Saved all places data to 'outputs/csv_files/data_all_places_dataset_id_342.csv'.
----
GeoJSON file created: outputs/geofiles/data_all_places_dataset_id_342.geojson
----
Shapefile created: outputs/geofiles/data_all_places_dataset_id_342.shp


/var/folders/mm/yfvypqbj1yq562m984qdwp4m0000gn/T/ipykernel_43244/2187217740.py:85: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(shapefile_path, driver="ESRI Shapefile")
/Users/jpgbs/Dev/CartoSpot/scripts/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'canopy_area' to 'canopy_are'
  ogr_write(
